In [6]:
from pathlib import Path
import tensorflow as tf
import pandas as pd
# -----------------------
# CONFIG
# -----------------------
INPUT_DIR = Path("/Users/erinekman/cb1020/Hud-cancer/bilder_projektcb1020") #Ändra till din egen path om man vill testa själv
OUTPUT_DIR = Path("/Users/erinekman/cb1020/Hud-cancer/bilder_ut")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
 
# -----------------------
# COPY CSV METADATA
# -----------------------
# Copies all CSV files from the parent folder (Notebook1)
# into the output folder together with the resized images

CSV_FILE = Path("/Users/erinekman/cb1020/Hud-cancer/bilder_projektcb1020/metadata.csv")

if CSV_FILE.exists():

    df = pd.read_csv(CSV_FILE)

    # Keep only these columns
    df_small = df[["isic_id", "diagnosis_1"]]

    # Save into output folder
    df_small.to_csv(
        OUTPUT_DIR / "metadata_reduced.csv",
        index=False
    )

    print("Created metadata_reduced.csv")

# -----------------------
# SETTINGS
# -----------------------
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

processed = 0
failed = 0




# -----------------------
# PROCESS ALL IMAGES
# -----------------------
for image_path in INPUT_DIR.rglob("*"):

    if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
        continue

    try:
        # Read image
        image = tf.io.read_file(str(image_path))

        # Decode image
        image = tf.image.decode_jpeg(
            image,
            channels=3
        )

        image = tf.image.resize_with_pad(
            image,
            224,
            224
        )
        
        #Normalisering, dela på 255 
        image = tf.cast(image, tf.float32) / 255.0 
        
        """ #Convert back to uint8 for storage, TA BORT VID SKARP TRÄNING!
        image = tf.cast(
            tf.clip_by_value(image, 0, 255),
            tf.uint8 
         )"""

        """# Save with EXACT same filename
        output_image = OUTPUT_DIR / image_path.name

        tf.io.write_file(
            str(output_image),
            tf.io.encode_jpeg(image)    #Ta bort (jpeg) och byt ut vid skarp träning?
        )"""

        processed += 1

        if processed % 500 == 0:
            print(f"Processed {processed} images")

    except Exception as e:
        failed += 1
        print(f"Failed: {image_path.name} -> {e}")

# -----------------------
# SUMMARY
# -----------------------
print("\nDone!")
print(f"Images processed: {processed-1}")
print(f"Errors: {failed}")


Created metadata_reduced.csv

Done!
Images processed: 15
Errors: 0
